# 🚨 Crisis Negotiator — Full GRPO Training on Colab

Runs Ollama locally on the Colab GPU, then trains the negotiator agent against the environment with a real LLM.

**Runtime:** T4 GPU (free tier) | **Time:** ~40 min for 100 episodes

In [ ]:
# Cell 1: Install Ollama + Python deps
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q openenv-core fastapi uvicorn pydantic openai requests numpy matplotlib

In [ ]:
# Cell 2: Start Ollama server and pull model
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:3b
print('Ollama ready with qwen2.5:3b')

In [ ]:
# Cell 3: Clone repo
!git clone https://github.com/Dinesh052/Test.git crisis-negotiator
%cd crisis-negotiator

In [ ]:
# Cell 4: Verify environment + Ollama connection
import sys, os
sys.path.insert(0, '.')
os.environ['API_BASE_URL'] = 'http://localhost:11434/v1'
os.environ['MODEL_NAME'] = 'qwen2.5:3b'
os.environ['HF_TOKEN'] = 'ollama'

# Test env
from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction
env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
obs = env.step(NegotiatorAction(action_type='emotional_label', content='I hear you.', reasoning='', target='hostage_taker'))
print(f'Environment OK: reward={obs.reward:.3f}')

# Test Ollama
from openai import OpenAI
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
r = client.chat.completions.create(model='qwen2.5:3b', messages=[{'role':'user','content':'Say hi'}], max_tokens=10)
print(f'Ollama OK: {r.choices[0].message.content}')

In [ ]:
# Cell 5: Run GRPO training (100 episodes with LLM + curriculum)
os.environ['MAX_STEPS'] = '100'
!python -u train_grpo.py

In [ ]:
# Cell 6: Plot reward curves
!python plot_rewards.py

In [ ]:
# Cell 7: Display the reward curve
from IPython.display import Image
Image('reward_curve.png')

In [ ]:
# Cell 8: Summary stats
import json
data = json.load(open('reward_log.json'))
rewards = [d['reward'] for d in data]
first_10 = rewards[:10]
last_10 = rewards[-10:]

print(f'Episodes: {len(data)}')
print(f'First 10 avg reward: {sum(first_10)/10:.3f}')
print(f'Last 10 avg reward:  {sum(last_10)/10:.3f}')
print(f'Improvement: {sum(last_10)/10 - sum(first_10)/10:+.3f}')
print(f'Overall success rate: {sum(1 for r in rewards if r>0.5)/len(rewards):.0%}')

# Oversight stats
unsafe = sum(1 for d in data if d['outcome'] in ['harm_event','tactical_intervention','supervisor_termination'])
predicted = sum(1 for d in data if d.get('oversight_f1', 0) > 0)
print(f'\nOversight: predicted {predicted}/{unsafe} dangerous situations')

In [ ]:
# Cell 9: Download artifacts
from google.colab import files
files.download('reward_curve.png')
files.download('reward_log.json')